<br>
<font>
<div dir=ltr align=center>
<br>
<font color=0F5298 size=8>
Housing price predict <br>
<font color= 6C3BAA size=6>
Machine Learning Project <br>
<font color=696880 size=5>
<!-- <br> -->
Shahid Beheshti University

<font color=GREEN size=5>
<br>
Alireza Hoseini
<!-- <br> -->



---



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## **Phase 1.1: Data Loading**
### **Reading the AmesHousing database**
In the following cell, the Ames Housing dataset is imported and stored using the Pandas library, followed by an inspection of its dimensions, specifically the number of rows and columns.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/mnalireza/Ameshousing_AI_project/refs/heads/main/AmesHousing.csv')

print("Dataset dimensions :")
print(f"Rows = {df.shape[0]} and Columns = {df.shape[1]}")
print('\n\n\n')

## **Phase 1.2: Preliminary Review**
</br>

We want to perform a preliminary inspection to understand the general structure of the dataset:

    head(): Displays the first 5 rows of the dataset; however, the desired number of rows can also be specified inside the parentheses.

    info(): Specifies structural information such as data types and the number of rows.

    describe(): Provides a statistical summary of numerical columns, including count, mean, standard deviation, minimum and maximum values, and quartiles.

    Identifying Missing Data: By examining the Non-Null Count column in the output of the info() command, the number of valid data entries and the presence of missing values are determined.


In [ ]:
print("---The first five rows of the dataset---\n".center(120))
df.index = range(1, len(df) + 1)
display(df.head())

print('\n\n\n')
print("---Column information and data types---\n")
df.info()

print('\n\n\n')
print("---Descriptive statistics of the variables---".center(120))
display(df.describe())

missing_counts = df.isnull().sum()
print('\n\n\n')
print("---count of missing data in each column---\n".center(120))
print(missing_counts)

## **Phase 1.3: Target Variable Analysis**
<br>
<br>

### **Histogram & Log Transformation**

In this section, we analyze the statistical behavior and distribution plot of the target variable (SalePrice) before and after log transformation.
<br>
<br>

*   Initial Data Status: The histogram of the selling price exhibits a significant right-skewness (Right-Skewed) with a skewness of about 1.74. This indicates that most houses were traded in a moderate price range, and a small number of properties with extremely high prices caused asymmetry, pulling the distribution away from normality.
<br>

*   Log Transformation (np.log1p): By applying this transformation, the skewness is dramatically reduced to about -0.01. The resulting chart becomes more symmetrical and falls within the normal range (between -0.5 and 0.5), taking on a standard bell-shaped distribution similar to a Gaussian curve, which helps improve the performance of regression models.
<br>
<br>
<br>

### **Gaussian normal distribution and skewness**

In this section, we explain the mathematical foundations of skewness, distribution normality, and its statistical formulas for the target variable (SalePrice):  
<br>

*   Skewness: Skewness is a measure of the asymmetry of a statistical distribution of data around its mean. The mathematical formula for calculating skewness (the standardized third moment) is as follows:
  $$Skew = E\left[\left(\frac{X - \mu}{\sigma}\right)^3\right] = \frac{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^3}{\left(\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2\right)^{3/2}}$$
In the initial housing price data, the skewness value is approximately 1.74, which indicates positive skewness or right-skewness; meaning the tail of the distribution is stretched toward larger values.  
<br>

*   Normal (Gaussian) Distribution: The normal distribution is recognized by the following probability density function:
  $$f(x) = \frac{1}{\sigma \sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x - \mu}{\sigma}\right)^2}$$
where $\mu$ is the mean and $\sigma$ is the standard deviation. The symmetry resulting from the log transformation brings the target variable closer to this bell-shaped structure and reduces the error of regression models.  



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df['SalePrice'], color='blue', ax=axes[0], bins=40)
axes[0].set_title('SalePrice', fontsize=14)
axes[0].set_xlabel('Selling price', fontsize=12)
axes[0].set_ylabel('Quantity', fontsize=12)

sns.histplot(np.log1p(df['SalePrice']), color='green', ax=axes[1], bins=40)
axes[1].set_title('log of SalePrice', fontsize=14)
axes[1].set_xlabel('Log(SalePrice + 1)', fontsize=12)
axes[1].set_ylabel('Quantity', fontsize=12)

plt.tight_layout()
plt.show()
initial_skewness = df['SalePrice'].skew()
log_skewness = np.log1p(df['SalePrice']).skew()
print(f"Skewness of the initial histogram : {initial_skewness:.2f}")
print(f"Skewness of the logarithmic histogram : {log_skewness:.2f}")
if (initial_skewness < 0.5 and initial_skewness > -0.5) :
  print("The initial histogram is normal.")
else :
  print("The initial histogram is unnormal.")
if (log_skewness < 0.5 and log_skewness > -0.5) :
  print("The logarithmic histogram is normal.")
else :
  print("The logarithmic histogram is unnormal.")

## **Phase 1.4: Missing Values**

### **Missing data management**

In this section, we handle the missing values in the dataset. For numerical columns, we fill the empty values with the median of that column, since the median is more resistant to outliers and is less affected by extreme values than the mean. For categorical columns, we replace missing values with the mode, i.e., the most frequent value in that column. This way, no row is dropped from the dataset due to missing data, and the overall distribution of the data remains largely intact.

In [ ]:
numerical = df.select_dtypes(include=[np.number]).columns.tolist()
categorical = df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in numerical:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in categorical:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values imputed successfully based on project instructions.")

## **Phase 1.5: One-Hot Encoding**
</br>
</br>

### **Categorical Data Encoding**

One-hot encoding is a way to turn categorical (text-based) data into numbers, since machine learning models can't work directly with things like a neighborhood name or a house style — they need numeric input.

The way it works: for each possible category, a new column is created with values of 0 or 1. If a row belongs to that category, it gets a 1; otherwise, it gets a 0. So if a "neighborhood" column had three possible values, it would normally create three new columns (or two, if we drop one).
</br>
</br>
</br>
### **Code Explanation**


*   That's actually the reason drop_first=True is used here: it drops one category from each column, which avoids what's called the "dummy variable trap" basically a situation where one column becomes redundant because you could already figure out its value just by looking at the others.

*   This line uses pd.get_dummies to convert the two categorical columns, Neighborhood and House Style, into one-hot encoded format. With drop_first=True, one category from each column is dropped as a reference to avoid redundancy, and with dtype=int, the resulting values are stored as 0 and 1 instead of True/False. The code then prints how many new features were generated by this encoding, and finally displays the first few rows of these new columns so the output structure can be inspected.



In [ ]:
encoded_cols = pd.get_dummies(df[['Neighborhood', 'House Style']], drop_first=True, dtype=int)
print(f"Number of new features generated by one-hot encoding : {encoded_cols.shape[1]}")
print('\n\n\n')
encoded_cols.index = range(1, len(encoded_cols) + 1)
encoded_cols.head()

## **Phase 2.1: Create a new feature**

This part is about feature engineering, creating new variables from the existing columns that can carry more useful information for the model than the raw columns alone.

First, we create TotalSF, which is the sum of the above-ground living area (Gr Liv Area) and the basement area (Total Bsmt SF). This feature represents the total usable square footage of the house and usually correlates much more strongly with the sale price than either of the two individual columns.

Second, we create HouseAge, calculated as the difference between the year sold (Yr Sold) and the year built (Year Built), which gives the age of the house at the time of sale. This feature typically has a meaningful relationship with price, since older houses tend to be cheaper.

The rest of the code displays a few relevant columns side by side for each of these two features, just to confirm that the new values were calculated correctly.

In [ ]:
df['TotalSF'] = df['Gr Liv Area'] + df['Total Bsmt SF']

df['HouseAge'] = df['Yr Sold'] - df['Year Built']



print("---Add Total SF feature---\n\n\n".center(52))
df_display_1 = df[['Gr Liv Area', 'Total Bsmt SF', 'TotalSF', 'SalePrice']]
df_display_1.index = range(1, len(df_display_1) + 1)
display(df_display_1.head())

print('\n\n\n')

print("---Add HouseAge feature---\n\n\n".center(35))
df_display = df[['Yr Sold', 'Year Built', 'HouseAge']]
df_display.index = range(1, len(df_display) + 1)
display(df_display.head())

## **Phase 2.2: Key Feature Selection**

In this cell, we create a variable called Feature that stores the 5 columns we selected from the dataset, and using the head() command, we display the first 5 rows of these columns in a table format.

In [ ]:
Feature = df[['Overall Qual', 'Gr Liv Area', 'Total Bsmt SF', 'Year Built', 'TotalSF']]

print("---Some features in a table---\n\n\n".center(64))
Feature.index = range(1, len(Feature) + 1)
display(Feature.head())

## **Plotting graphs**
</br>
</br>

In each of the 5 cells below, a chart is plotted for each of the feature columns against the sale price, and the type of chart was chosen based on the characteristics of each column so we'd get the best possible visualization.
</br>
</br>

### **First cell :**

For the first chart, which measures overall quality against sale price, I used a violin plot. The reason for using it is that overall quality is a discrete quantity from 1 to 10, while the sale price is continuous, and it shows, for each quality level, around what price most of the sold houses fall at.

In other words, the distinctive feature of this type of chart is that it shows the density of the price.
</br>
</br>

### **Second cell :**

For the second chart, I used a line chart because it showed the year the house was built against the sale price. This isn't a linear relationship where we'd want to use a scatter plot for it.

Using a line chart makes it easier to show the approximate price range for houses with different years built.
</br>
</br>

### **The last three cells :**
For the last three charts, I used scatter plots because in all three, they showed the square footage of a part of the house against the sale price, which is roughly a linear relationship and can be evaluated with linear regression.

Using a scatter plot makes it easier to show the approximate price range for houses with different square footages.

In [ ]:
plt.figure(figsize=(14, 6))
sns.violinplot(data=df, x='Overall Qual', y='SalePrice', palette='magma', inner='quartile', hue='Overall Qual', legend=False)

plt.title('Price Density Across Quality Levels', fontsize=14)
plt.xlabel('Overall Quality', fontsize=12)
plt.ylabel('Sale Price', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=df, x='Year Built', y='SalePrice', linewidth=3)

plt.title('Trend of Sale Price by Year Built', fontsize=14)
plt.xlabel('Year Built', fontsize=12)
plt.ylabel('Sale Price', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.regplot(data=df, x='Gr Liv Area', y='SalePrice', line_kws={'color': '#d62728'})

plt.title('Relationship Between Grade Living Area and Sale Price', fontsize=14)
plt.xlabel('Above Grade Living Area', fontsize=12)
plt.ylabel('Sale Price', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.regplot(data=df, x='Total Bsmt SF', y='SalePrice', scatter_kws={'color': '#2ca02c'}, line_kws={'color': '#800080'})

plt.title('Relationship Between Total Basement Square Feet and Sale Price', fontsize=14)
plt.xlabel('Total Basement Square Feet', fontsize=12)
plt.ylabel('Sale Price', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.regplot(data=df, x='TotalSF', y='SalePrice', scatter_kws={'color': '#E5A93C'}, line_kws={'color': '#5A6B7C'})

plt.title('Relationship Between Total Square Feet and Sale Price', fontsize=14)
plt.xlabel('Total Square Feet', fontsize=12)
plt.ylabel('Sale Price', fontsize=12)
plt.tight_layout()
plt.show()

## **Phase 2.3 : Visualization of the relationships**
### **Scatter Plot**
Here, I plotted the scatter plot of all 5 features against the sale price side by side.

As can be seen from the points in the charts, plotting a scatter plot isn't very suitable for overall qual and year built, since no good linear relationship is visible between them.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, feature in enumerate(Feature):
    sns.scatterplot(data=df, x=feature, y='SalePrice', ax=axes[i], color='blue')
    sns.regplot(data=df, x=feature, y='SalePrice', ax=axes[i], scatter=False, color='red')
    axes[i].set_title(f'Realationship between {feature} and Sale Price', fontsize=14)
    axes[i].set_xlabel(feature, fontsize=11)
    axes[i].set_ylabel('SalePrice', fontsize=11)
fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

## **Coefficient**
In the cell below, the regression coefficient for each scatter plot is displayed.

However, it should be noted that this is beyond the scope of the project.

In [ ]:
coef_data = []
for f in Feature.columns:
    simple_model = LinearRegression()
    simple_model.fit(df[[f]], df['SalePrice'])
    coef_data.append({
        'Feature': f,
        'Coefficient': simple_model.coef_[0]
    })

coef = pd.DataFrame(coef_data)
coef.index = range(1, len(coef) + 1)
print("---Comparison of multivariate vs univariate coefficients---\n\n\n")
display(coef.style.format({'Coefficient': '{:.2f}'}))

## **Phase 3.1 : Data Sepration**
In this cell, we split the data that was in the 5 feature columns into training data and test data, putting 80% of the data into the training set and 20% into the test set.

We do the same for the saleprice data as well, so that we can use a linear regression model to predict saleprice from the features.

We do this using the train_test_split command, which randomly splits the data into two sets: test data and training data.

The test_size=0.2 argument is used so the data gets split in an 80% to 20% ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Feature, df['SalePrice'], test_size=0.20, random_state=42)

print(f"تعداد داده‌های آموزش: {X_train.shape[0]}")
print(f"تعداد داده‌های آزمون: {X_test.shape[0]}")

## **Phase 3.2 : Model Definition and Selection**

In [ ]:
model = LinearRegression()

## **Phase 3.3: Model Training**

In the cell below, I fitted the model. This trains the linear regression model, meaning the model is learning from the training data how to establish a relationship between the features (X) and the house price (y).
</br>
</br>
</br>

### **Functioning of the fit Method in Linear Regression**
This method constitutes the core of the model training process. In linear regression, the fit method attempts to find the best line (or in multidimensional spaces, the best plane or hyperplane) that minimizes the distance between the actual values and the predicted values. This is typically accomplished using the Ordinary Least Squares (OLS) method; that is, the model determines coefficients such that the sum of squared errors (the difference between actual price and predicted price) is minimized.

Mathematically speaking, the model seeks to calculate the coefficients $\beta_1, \beta_2, \beta_3, \beta_4, \beta_5$ and the intercept $\beta_0$ in the following equation so that the total error is minimized:$$\text{SalePrice} = \beta_0 + \beta_1 \cdot (\text{Overall Qual}) + \beta_2 \cdot (\text{Gr Liv Area}) + \beta_3 \cdot (\text{Total Bsmt SF}) + \beta_4 \cdot (\text{Year Built}) + \beta_5 \cdot (\text{TotalSF})$$
</br>
</br>
</br>

### **Under-the-Hood Execution Steps**

*   The model receives the input data, namely X_train and y_train.

*   Using numerical methods (typically normal equations or SVD decomposition), it computes the optimal value for each feature coefficient (model.coef_) and the intercept (model.intercept_).
*   These calculated values are stored inside the model object itself. Consequently, after running this line, model is no longer a "raw" object, but a "trained" model that can be used for making predictions (model.predict(...)).




In [ ]:
model.fit(X_train, y_train)

## **Phase 4.1,2 : Model Evaluation**
</br>
</br>

### **Price Prediction (y_pred)**

By calling the model.predict(X_test) method, the trained model receives the feature values of the 20% test data and calculates the estimated price for each house. These values are stored in the y_pred variable to be compared against the actual prices, y_test.
</br>
</br>
</br>

### **Evaluation Metrics**



*   Root Mean Squared Error (RMSE):
This metric first squares the differences between actual and predicted prices, computes their mean, and finally takes the square root. Because the differences are squared, it penalizes larger errors more heavily. The unit of RMSE is the same as the target variable (dollars), straightforwardly showing how far the predictions deviate from the actual prices on average.

*   Mean Absolute Error (MAE):
This index calculates the average of the absolute differences between actual and predicted values. Unlike RMSE, it treats all errors with equal weight and is less sensitive to outliers. MAE indicates the average dollar error the model makes per house in its price estimation.


*   Coefficient of Determination (R-squared / $R^2$ Score):
This metric indicates what percentage of the variance and changes in house prices (SalePrice) can be explained and predicted by the model's input features. Its value scales between 0 and 1 (or as a percentage from 0% to 100%); the closer this number is to 1, the greater the model's ability to explain data relationships and make accurate predictions.




In [ ]:
y_pred= model.predict(X_test)

RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
MAE = mean_absolute_error(y_test, y_pred)
acc = r2_score(y_test, y_pred)

print("---Linear regression model evaluation results on the test set---\n\n\n")

print(f"Root Mean Squared Error : ${RMSE:,.2f}\n")
print(f"Mean Absolute Error : ${MAE:,.2f}\n")
print(f"R-squared Score (R2) : {acc*100:.2f}%")

## **Phase 4.3 : Interpretation of Results**

### **Comprehensive Model Performance Analysis and Evaluation**

In this cell, I prepared a complete performance report for my model on the test data:

*   Evaluating Key Metrics: I first calculated the average actual house price and measured the ratio of the Mean Absolute Error to the mean price (MAE_percent) to gain a more intuitive understanding of my model's error percentage.

*   Analyzing Outlier Effects (RMSE vs. MAE): Since RMSE is more sensitive to large errors due to squaring the differences, I calculated the $\frac{\text{RMSE}}{\text{MAE}}$ ratio to examine the gap between the two. This ratio indicates whether my model produces significantly larger errors on unique or luxury properties.
*   Top 10 Model Errors Table: Finally, by constructing a DataFrame containing actual prices, predicted prices, and percentage errors, I extracted and displayed the top 10 records with the highest percentage errors to closely examine the potential reasons behind accuracy drops on these specific cases.


In [ ]:
mean_actual_price = y_test.mean()
MAE_percent = (MAE / mean_actual_price) * 100

BOLD = '\033[1m'
UNDERLINE = '\033[4m'
RESET = '\033[0m'

title = f"{BOLD}{UNDERLINE}LINEAR REGRESSION PERFORMANCE REPORT{RESET}"
print(title.center(70))
print("\n\n\n")

print(f"Mean Actual House Price: ${mean_actual_price:,.2f}")
print(f"Mean Absolute Error (MAE): ${MAE:,.2f} ({MAE_percent:.2f}% of mean price)")
print(f"Root Mean Squared Error (RMSE): ${RMSE:,.2f}")
print(f"Model accuracy : {acc*100:.2f}%")
print(f"Model eror : {MAE_percent:.2f}%\n\n\n\n")

if RMSE > MAE:
    ratio = RMSE / MAE
    print(f"---Analysis of Outlier Effects (RMSE vs MAE)---\n\n".center(80))
    print(f"The RMSE is approximately {ratio:.2f} times the MAE.")
    print(
        "This difference is due to the higher sensitivity of squaring in RMSE to"
        " large errors,"
    )
    print(
        "indicating that the model incurs larger errors for very large properties"
        " or luxury homes.\n\n\n\n"
    )

Eror_Percent = (np.abs(y_test - y_pred) / y_test ) * 100
error_analysis = pd.DataFrame({'Actual Price': y_test, 'Predicted Price': y_pred, 'Absolute Error': np.abs(y_test - y_pred), 'Eror Percent': Eror_Percent}, index=y_test.index)

top_errors = error_analysis.sort_values(by='Eror Percent', ascending=False).head(10)
top_errors.index = range(1, len(top_errors) + 1)

print("---Top 10 Higher eror---\n".center(65))
display(top_errors.style.format({'Actual Price': '${:,.2f}','Predicted Price': '${:,.2f}','Absolute Error': '${:,.2f}','Eror Percent': '{:,.2f}%'}))

<br>
<font>
<div dir=ltr align=center>
<font color= 6C3BAA size=8>
Bonus Challenges <br>

## **Chsallenge 1 : RF and GB Regression**

In this cell, to better assess the capability of the linear regression model and explore the potential presence of non-linear relationships in the data, I implemented and trained two robust ensemble learning models: Random Forest Regressor and Gradient Boosting Regressor, each configured with 100 estimators (n_estimators=100):



*   Training and Evaluating Ensemble Models: I fitted both models on the training sets (X_train and y_train) and evaluated their predictions on the test set (X_test), computing key metrics—namely RMSE, MAE, and the $R^2$ Score—for each algorithm individually.

</br>
</br>

### **Did the results improve?**

Yes, comparing the results reveals that deploying more complex ensemble models such as Random Forest Regressor and Gradient Boosting Regressor substantially improves predictive performance over Linear Regression:



*   **Reduction in Prediction Errors:** Both dollar based error metrics RMSE and MAE decreased significantly in the ensemble models compared to linear regression. This demonstrates that the more complex models generated predictions closer to actual selling prices on average and handled price anomalies and extreme values with greater flexibility.

*   **Higher Coefficient of Determination ($R^2$ Score):** The $R^2$ score increased noticeably under Random Forest and Gradient Boosting, indicating that these models account for a larger share of the total variance and fluctuations in housing prices.

*   **Reason for the Improvement:** Linear regression inherently assumes a strictly linear and additive relationship between the features (such as living area or year built) and house price. In contrast, real-world real estate pricing involves non-linear dynamics and feature interactions. Tree-based ensemble algorithms successfully capture these high-dimensional patterns and threshold-based behaviors without requiring manual interaction engineering.


In [ ]:
RF_model = RandomForestRegressor(n_estimators=100, random_state=42)
RF_model.fit(X_train, y_train)
y_pred_RF = RF_model.predict(X_test)

RMSE_RF = np.sqrt(mean_squared_error(y_test, y_pred_RF))
MAE_RF = mean_absolute_error(y_test, y_pred_RF)
R2_RF = r2_score(y_test, y_pred_RF)

GB_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
GB_model.fit(X_train, y_train)
y_pred_GB = GB_model.predict(X_test)

RMSE_GB = np.sqrt(mean_squared_error(y_test, y_pred_GB))
MAE_GB = mean_absolute_error(y_test, y_pred_GB)
R2_GB = r2_score(y_test, y_pred_GB)

models_compare = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor', 'Gradient Boosting Regressor'],
    'RMSE': [RMSE, RMSE_RF, RMSE_GB],
    'MAE': [MAE, MAE_RF, MAE_GB],
    'R² Score': [acc*100, R2_RF*100, R2_GB*100]
})

print("---Performance Comparison of Models on Test Data---\n".center(55))
models_compare.index = range(1, len(models_compare) + 1)
models_compare = models_compare.style.format({'RMSE': '${:,.2f}', 'MAE': '${:,.2f}', 'R² Score': '{:.2f}%'})
display(models_compare)


## **Challenge 2 : Feature Standardization with StandardScaler**

### **Data Standardization and Its Impact on Different Models**
In this section, I standardized the input features using StandardScaler (scaling them to zero mean and unit variance). I scaled the training data using fit_transform and applied only transform to the test data to prevent data leakage.
</br>
</br>
</br>

### **Results in Ordinary Least Squares (OLS) Linear Regression**



*   **Unchanged Accuracy and $R^2$:** As seen in the output, standardization causes no change in the final accuracy, RMSE, or $R^2$ score of standard linear regression. This is because OLS applies a linear transformation to the coefficients, directly compensating for changes in feature scale via proportional adjustments in coefficient magnitudes.

*   **Interpretability and Feature Importance:** The primary goal of standardizing in linear regression is to make the coefficients directly comparable. Normally, a feature with large numerical values (such as square footage) receives a very small coefficient, while a feature on a small scale (such as build quality rated 1 to 10) receives a large coefficient. Standardization removes the effect of physical measurement units, meaning each standardized coefficient represents the dollar change in the target variable for every one standard deviation change in that feature. Consequently, features with the highest absolute coefficients can be ranked as the most influential drivers of price.



In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaled_model = LinearRegression()
scaled_model.fit(X_train_scaled, y_train)
y_pred_scaled = scaled_model.predict(X_test_scaled)

RMSE_scaled = np.sqrt(mean_squared_error(y_test, y_pred_scaled))
R2_scaled = r2_score(y_test, y_pred_scaled)

print(f"R² Before standardization : {acc:.4f}")
print(f"R² After standardization : {R2_scaled:.4f}")

standardized_coefs = pd.DataFrame({
    'Feature': Feature.columns,
    'Standardized_Coefficient': scaled_model.coef_
}).sort_values(by='Standardized_Coefficient', key=abs, ascending=False)

print("اهمیت متغیرها بر اساس ضرایب استانداردشده (تأثیر هر یک انحراف معیار تغییر بر قیمت):")
display(standardized_coefs.style.format({'Standardized_Coefficient': '{:.0f}'}))

## **Challenge 3 : Residuals Analysis**

In this section, to validate the key assumptions of linear regression and examine how prediction errors are distributed, the residuals are visualized using two complementary plots:
</br>
</br>
</br>

### **Distribution of Residuals:**


*   The error distribution (histogram with KDE overlay) is centered around zero and roughly resembles a bell curve, validating the normality assumption for the vast majority of observations.

*   Nevertheless, a prominent right tail (positive skewness) is visible, indicating that extreme outliers and high-value properties produce large positive residuals.


In [ ]:
Residuals = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize = (16, 6))
sns.scatterplot(x = y_pred, y = Residuals, color='orange', ax = axes[0], edgecolors='k')
axes[0].axhline(y=0, color='navy', linestyle='--', linewidth=1.5)
axes[0].set_title('Residuals vs Predicted', fontsize=14)
axes[0].set_xlabel('Predicted Values', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)

sns.histplot(Residuals, kde=True, color='crimson', ax = axes[1], bins=40)
axes[1].set_title('Distribution of Residuals', fontsize=14)
axes[1].set_xlabel('Residuals', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)

plt.tight_layout()
plt.show()